# 🦷 Dental Disease Risk Assessment — End-to-End ML Training Notebook
### Research Pipeline | Model 1: Risk Score Regression · Model 2: Risk Level Classification · Model 3: Care Plan Prediction

---

**Pipeline Overview:**

| Step | What happens |
|------|-------------|
| Section 1 | Install & import all libraries |
| Section 2 | Load the dataset |
| Section 3 | Exploratory Data Analysis (EDA) |
| Section 4 | Feature Engineering & Encoding |
| Section 5 | Train/Test Split |
| Section 6 | Model 1 — Risk Score Regression (XGBoost) |
| Section 7 | Model 2 — Risk Level Classification (Random Forest) |
| Section 8 | Model 3 — Care Plan Prediction (LightGBM MultiOutput) |
| Section 9 | Save all models & encoders |
| Section 10 | End-to-end Prediction Pipeline |

> ⚠️ **Research Disclaimer:** Synthetic data only. Not real clinical data. Does not replace a licensed dentist.


## 📦 Section 1 — Install & Import Libraries

In [1]:
!pip install pandas numpy matplotlib seaborn scikit-learn xgboost lightgbm joblib imbalanced-learn

In [2]:
pip install lightgbm

Note: you may need to restart the kernel to use updated packages.


In [3]:
import os, json, warnings
warnings.filterwarnings("ignore")          # suppress verbose sklearn/xgb warnings

import numpy  as np
import pandas as pd

import matplotlib
matplotlib.use("Agg")                      # non-interactive backend for saving figures
import matplotlib.pyplot as plt
import seaborn as sns

# ── Preprocessing ──────────────────────────────────────────────────────────
from sklearn.preprocessing   import LabelEncoder
from sklearn.model_selection  import train_test_split, cross_val_score
from imblearn.over_sampling   import SMOTE            # handles class imbalance for Model 2

# ── Models ──────────────────────────────────────────────────────────────────
from sklearn.ensemble         import RandomForestClassifier, RandomForestRegressor
from sklearn.linear_model     import LogisticRegression, LinearRegression
from sklearn.tree             import DecisionTreeClassifier
from sklearn.multioutput      import MultiOutputClassifier
from xgboost                  import XGBRegressor, XGBClassifier
import lightgbm               as lgb
from lightgbm                 import LGBMClassifier

# ── Metrics ─────────────────────────────────────────────────────────────────
from sklearn.metrics import (
    mean_squared_error, mean_absolute_error, r2_score,       # regression metrics
    accuracy_score, f1_score, classification_report,         # classification metrics
    confusion_matrix
)

import joblib                              # for saving/loading trained models

# Create output directories to store graphs and saved models
os.makedirs("outputs/graphs", exist_ok=True)
os.makedirs("outputs/models", exist_ok=True)

sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams.update({"figure.dpi": 120, "axes.titlesize": 13})

print("✅ All libraries loaded successfully.")
print(f"   LightGBM version : {lgb.__version__}")

✅ All libraries loaded successfully.
   LightGBM version : 4.6.0


## 📂 Section 2 — Load Dataset

The dataset has **10,000 patient records** with:
- **18 input features** (age, habits, conditions, exam findings)
- **1 continuous target** → `risk_score` (0–100) for Model 1
- **9 care plan targets** (toothpaste, mouthwash, urgency, etc.) for Model 3
- `risk_level` is **derived** from `risk_score` → used for Model 2


In [4]:
# ── Load dataset ─────────────────────────────────────────────────────────────
CSV_PATH = "data/dental_dataset.csv"   

df = pd.read_csv(CSV_PATH)

print(f"Dataset shape  : {df.shape}  ({df.shape[0]} patients × {df.shape[1]} columns)")
print(f"Missing values : {df.isnull().sum().sum()}")   # confirm no nulls
print()
print("Column overview:")
print(df.dtypes)
print()
df.head(4)

Dataset shape  : (10000, 28)  (10000 patients × 28 columns)
Missing values : 0

Column overview:
age                                int64
number_of_teeth                    int64
number_of_missing_teeth            int64
is_primary_teeth                    bool
smoking_status                       str
alcohol_usage                        str
sugar_usage                          str
brushing_frequency                 int64
diabetes_status                     bool
pregnancy_status                    bool
gum_bleeding                        bool
tooth_sensitivity                   bool
calcium_or_vitamin_deficiency       bool
number_of_filled_teeth             int64
overall_oral_hygiene_level           str
identified_disease                   str
disease_severity_from_xray           str
affected_teeth_count               int64
risk_score                       float64
toothpaste_type                      str
mouthwash_type                       str
interdental_tool                     str
r

,age,number_of_teeth,number_of_missing_teeth,is_primary_teeth,smoking_status,alcohol_usage,sugar_usage,brushing_frequency,diabetes_status,pregnancy_status,...,risk_score,toothpaste_type,mouthwash_type,interdental_tool,rinse_type,pain_relief,dietary_action,dentist_urgency,monitoring_focus,lifestyle_action
0,40,25,1,False,medium,no,medium,1,True,False,...,91.62,tartar-control-fluoride,antiseptic-chlorhexidine,both,chlorhexidine-rinse,nsaid-plus-topical,no-change,1-week,multiple-symptoms,smoking-reduction
1,24,24,0,False,no,no,no,1,False,True,...,35.40,standard-fluoride,fluoride-rinse,none,none,none,no-change,3-months,routine,none
2,8,13,3,True,no,no,medium,2,False,False,...,90.26,standard-fluoride,none,none,chlorhexidine-rinse,paracetamol-type,no-change,1-week,multiple-symptoms,none
3,54,21,8,False,no,medium,medium,1,True,False,...,63.24,standard-fluoride,antibacterial,interdental-brush-only,none,none,reduce-acid,4-weeks,routine,alcohol-reduction


In [5]:
# ── Derive risk_level from risk_score ────────────────────────────────────────
# risk_score is a continuous 0–100 value in the dataset.
# We bucket it into 3 ordinal categories for Model 2 (classification).
#
#   Low    → score  0–40   (mild conditions, good hygiene)
#   Medium → score 41–70   (moderate risk factors present)
#   High   → score 71–100  (severe conditions, multiple risk factors)

df["risk_level"] = pd.cut(
    df["risk_score"],
    bins=[0, 40, 70, 100],
    labels=["low", "medium", "high"]
)

print("risk_level class distribution:")
print(df["risk_level"].value_counts())
print()
print("This is our Model 2 classification target (Low / Medium / High).")

risk_level class distribution:
risk_level
medium    5657
low       2301
high      2042
Name: count, dtype: int64

This is our Model 2 classification target (Low / Medium / High).


## 📊 Section 3 — Exploratory Data Analysis (EDA)

Visualise the key distributions before training to understand the data shape.


In [6]:
# ── Fig 1: Risk Score Distribution ───────────────────────────────────────────
# Shows whether risk_score is normally distributed — important for regression.
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

axes[0].hist(df["risk_score"], bins=40, color="#2980b9", edgecolor="white", alpha=0.85)
axes[0].axvline(df["risk_score"].mean(),   color="red",    linestyle="--", label=f"Mean:   {df['risk_score'].mean():.1f}")
axes[0].axvline(df["risk_score"].median(), color="orange", linestyle="--", label=f"Median: {df['risk_score'].median():.1f}")
axes[0].set_title("Risk Score Distribution")
axes[0].set_xlabel("Risk Score (0–100)")
axes[0].set_ylabel("Frequency")
axes[0].legend()

# Risk level pie
counts = df["risk_level"].value_counts()
axes[1].pie(counts, labels=counts.index, autopct="%1.1f%%",
            colors=["#2ecc71","#f39c12","#e74c3c"], startangle=140)
axes[1].set_title("Risk Level Class Balance")

plt.suptitle("Risk Score & Risk Level Overview", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig("outputs/graphs/01_risk_overview.png", bbox_inches="tight")
plt.show()
print("Saved → outputs/graphs/01_risk_overview.png")

Saved → outputs/graphs/01_risk_overview.png


In [7]:
# ── Fig 2: Key Feature Distributions ─────────────────────────────────────────
# the spread of the most important input features.
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
axes = axes.flatten()

num_features = ["age", "number_of_teeth", "number_of_missing_teeth",
                "affected_teeth_count", "number_of_filled_teeth", "brushing_frequency"]

for ax, col in zip(axes, num_features):
    ax.hist(df[col], bins=25, color="#8e44ad", edgecolor="white", alpha=0.8)
    ax.set_title(col.replace("_", " ").title())
    ax.set_xlabel("Value")
    ax.set_ylabel("Count")

plt.suptitle("Numeric Feature Distributions", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig("outputs/graphs/02_numeric_distributions.png", bbox_inches="tight")
plt.show()

In [8]:
# ── Fig 3: Risk Score vs Key Features ────────────────────────────────────────
# Scatter plots reveal whether individual features correlate with risk_score.
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

pairs = [
    ("age",                  "#2980b9"),
    ("affected_teeth_count", "#e74c3c"),
    ("number_of_filled_teeth","#27ae60"),
]

for ax, (col, color) in zip(axes, pairs):
    ax.scatter(df[col], df["risk_score"], alpha=0.15, s=8, color=color)
    # Add trend line
    z = np.polyfit(df[col], df["risk_score"], 1)
    p = np.poly1d(z)
    xs = np.linspace(df[col].min(), df[col].max(), 100)
    ax.plot(xs, p(xs), "r--", linewidth=1.5, label="trend")
    ax.set_title(f"risk_score vs {col.replace('_',' ')}")
    ax.set_xlabel(col.replace("_", " ").title())
    ax.set_ylabel("Risk Score")
    ax.legend()

plt.suptitle("Risk Score vs Numeric Features", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig("outputs/graphs/03_risk_score_vs_features.png", bbox_inches="tight")
plt.show()

In [9]:
# ── Fig 4: Categorical Feature vs Risk Level ─────────────────────────────────
# Bar charts show how categorical habits relate to the derived risk level.
cat_features = ["smoking_status", "alcohol_usage", "sugar_usage",
                "overall_oral_hygiene_level", "disease_severity_from_xray", "identified_disease"]

fig, axes = plt.subplots(2, 3, figsize=(16, 9))
axes = axes.flatten()

for ax, col in zip(axes, cat_features):
    ct = pd.crosstab(df[col], df["risk_level"])
    # Reorder risk level columns left-to-right
    for lvl in ["low","medium","high"]:
        if lvl not in ct.columns:
            ct[lvl] = 0
    ct = ct[["low","medium","high"]]
    ct.plot(kind="bar", ax=ax, color=["#2ecc71","#f39c12","#e74c3c"],
            edgecolor="white", legend=(ax == axes[0]))
    ax.set_title(col.replace("_"," ").title())
    ax.set_xlabel("")
    ax.set_ylabel("Patient Count")
    ax.tick_params(axis="x", rotation=15)

plt.suptitle("Categorical Features vs Risk Level", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig("outputs/graphs/04_categorical_vs_risk_level.png", bbox_inches="tight")
plt.show()

In [10]:
# ── Fig 5: Care Plan Target Distributions ────────────────────────────────────
# Shows the class balance for each of the 9 care plan outputs.
# Imbalanced targets may affect Model 3 performance on rare classes.
TARGET_COLS = [
    "toothpaste_type", "mouthwash_type",   "interdental_tool", "rinse_type",
    "pain_relief",     "dietary_action",   "dentist_urgency",  "monitoring_focus",
    "lifestyle_action"
]

fig, axes = plt.subplots(3, 3, figsize=(16, 13))
axes = axes.flatten()
colors = ["#2980b9","#27ae60","#8e44ad","#e67e22","#c0392b","#16a085","#d35400","#2c3e50","#7f8c8d"]

for ax, col, color in zip(axes, TARGET_COLS, colors):
    vc = df[col].value_counts()
    bars = ax.barh(vc.index, vc.values, color=color, alpha=0.85, edgecolor="white")
    ax.set_title(col.replace("_"," ").title(), fontsize=11, fontweight="bold")
    ax.set_xlabel("Count")
    for bar, val in zip(bars, vc.values):
        ax.text(bar.get_width()+30, bar.get_y()+bar.get_height()/2,
                f"{val:,}", va="center", fontsize=8)

plt.suptitle("Care Plan Target Variable Distributions", fontsize=14, fontweight="bold", y=1.01)
plt.tight_layout()
plt.savefig("outputs/graphs/05_careplan_target_distributions.png", bbox_inches="tight")
plt.show()
print("Saved → outputs/graphs/05_careplan_target_distributions.png")

Saved → outputs/graphs/05_careplan_target_distributions.png


## ⚙️ Section 4 — Feature Engineering & Encoding

**Why we encode:**
- Tree models (XGBoost, LightGBM, Random Forest) require numeric inputs.
- Ordinal features (e.g. `smoking: no < medium < high`) need ordered numbers, not one-hot.
- Boolean columns are already True/False → cast to 0/1.
- We derive 6 composite features that capture clinical domain knowledge.


In [11]:
# ── Working copy — never modify the original df ──────────────────────────────
df_enc = df.copy()

# ── 1. Ordinal encoding ───────────────────────────────────────────────────────
# Map ordered categories to integers that preserve the natural order.
# This is better than one-hot here because the ordering carries meaning.
ordinal_maps = {
    "smoking_status":             {"no": 0, "medium": 1, "high": 2},
    "alcohol_usage":              {"no": 0, "medium": 1, "high": 2},
    "sugar_usage":                {"no": 0, "medium": 1, "high": 2},
    "overall_oral_hygiene_level": {"good": 0, "moderate": 1, "poor": 2},
    "disease_severity_from_xray": {"mild": 0, "moderate": 1, "severe": 2},
}

for col, mapping in ordinal_maps.items():
    df_enc[col] = df_enc[col].map(mapping)
    print(f"  Encoded '{col}' → {mapping}")

print()

# ── 2. Binary encoding ────────────────────────────────────────────────────────
# identified_disease has 2 values → 1 binary column is enough (no info lost).
df_enc["identified_disease"] = (df_enc["identified_disease"] == "periodontal_bone_loss").astype(int)
print("  identified_disease → 0=dental_cavity, 1=periodontal_bone_loss")

# Boolean columns: Python True/False → 0/1
bool_cols = ["is_primary_teeth", "diabetes_status", "pregnancy_status",
             "gum_bleeding", "tooth_sensitivity", "calcium_or_vitamin_deficiency"]
for col in bool_cols:
    df_enc[col] = df_enc[col].astype(int)
print(f"  Boolean columns cast to int: {bool_cols}")

  Encoded 'smoking_status' → {'no': 0, 'medium': 1, 'high': 2}
  Encoded 'alcohol_usage' → {'no': 0, 'medium': 1, 'high': 2}
  Encoded 'sugar_usage' → {'no': 0, 'medium': 1, 'high': 2}
  Encoded 'overall_oral_hygiene_level' → {'good': 0, 'moderate': 1, 'poor': 2}
  Encoded 'disease_severity_from_xray' → {'mild': 0, 'moderate': 1, 'severe': 2}

  identified_disease → 0=dental_cavity, 1=periodontal_bone_loss
  Boolean columns cast to int: ['is_primary_teeth', 'diabetes_status', 'pregnancy_status', 'gum_bleeding', 'tooth_sensitivity', 'calcium_or_vitamin_deficiency']


In [12]:
# ── 3. Derived (engineered) features ─────────────────────────────────────────
# These capture clinically meaningful ratios and aggregated risk signals
# that a single raw column cannot express on its own.

total_teeth = df_enc["number_of_teeth"] + df_enc["number_of_missing_teeth"]

# Proportion of teeth missing — more meaningful than the raw count
df_enc["missing_teeth_ratio"] = (
    df_enc["number_of_missing_teeth"] / total_teeth.clip(lower=1)
)

# Proportion of remaining teeth that are filled — indicates history of decay
df_enc["filled_teeth_ratio"] = (
    df_enc["number_of_filled_teeth"] / df_enc["number_of_teeth"].clip(lower=1)
)

# Proportion of teeth affected by current disease
df_enc["affected_teeth_ratio"] = (
    df_enc["affected_teeth_count"] / df_enc["number_of_teeth"].clip(lower=1)
)

# Composite lifestyle risk: sum of smoking + alcohol + sugar (each 0/1/2)
# Range: 0 (no risk) → 6 (all three at high level)
df_enc["lifestyle_risk_score"] = (
    df_enc["smoking_status"] + df_enc["alcohol_usage"] + df_enc["sugar_usage"]
)

# Composite oral hygiene risk: combines hygiene level, brushing, bleeding, sensitivity
df_enc["oral_hygiene_risk_score"] = (
    df_enc["overall_oral_hygiene_level"] +
    (2 - df_enc["brushing_frequency"]) +   # invert: brushing 2x/day is LESS risk
    df_enc["gum_bleeding"] +
    df_enc["tooth_sensitivity"]
)

# Composite medical risk: diabetes + pregnancy + vitamin deficiency + primary teeth
df_enc["medical_risk_score"] = (
    df_enc["diabetes_status"] +
    df_enc["pregnancy_status"] +
    df_enc["calcium_or_vitamin_deficiency"] +
    df_enc["is_primary_teeth"]
)

print("✅ 6 derived features added:")
print("   missing_teeth_ratio, filled_teeth_ratio, affected_teeth_ratio")
print("   lifestyle_risk_score, oral_hygiene_risk_score, medical_risk_score")

# ── Final feature list used by ALL three models ────────────────────────────────
FEATURE_COLS = [
    # Raw demographics & counts
    "age", "number_of_teeth", "number_of_missing_teeth", "is_primary_teeth",
    # Lifestyle habits (ordinal encoded)
    "smoking_status", "alcohol_usage", "sugar_usage", "brushing_frequency",
    # Medical conditions
    "diabetes_status", "pregnancy_status", "gum_bleeding", "tooth_sensitivity",
    "calcium_or_vitamin_deficiency",
    # Clinical exam findings
    "number_of_filled_teeth", "overall_oral_hygiene_level", "identified_disease",
    "disease_severity_from_xray", "affected_teeth_count",
    # Engineered features
    "missing_teeth_ratio", "filled_teeth_ratio", "affected_teeth_ratio",
    "lifestyle_risk_score", "oral_hygiene_risk_score", "medical_risk_score",
]

print(f"\nTotal input features : {len(FEATURE_COLS)}")

✅ 6 derived features added:
   missing_teeth_ratio, filled_teeth_ratio, affected_teeth_ratio
   lifestyle_risk_score, oral_hygiene_risk_score, medical_risk_score

Total input features : 24


In [13]:
# ── Encode the 9 care plan TARGET columns for Model 3 ─────────────────────────
# LightGBM needs integer class labels (0, 1, 2...), not strings.
# We save each LabelEncoder so we can decode predictions back to text later.

label_encoders = {}

for col in TARGET_COLS:
    le = LabelEncoder()
    df_enc[col + "_enc"] = le.fit_transform(df_enc[col])
    label_encoders[col]  = le
    print(f"  {col:<25}: {len(le.classes_)} classes → {list(le.classes_)}")

TARGET_ENC_COLS = [t + "_enc" for t in TARGET_COLS]

# ── Encode risk_level for Model 2 ─────────────────────────────────────────────
risk_level_map = {"low": 0, "medium": 1, "high": 2}
df_enc["risk_level_enc"] = df_enc["risk_level"].map(risk_level_map)
print(f"\n  risk_level encoded: {risk_level_map}")

  toothpaste_type          : 4 classes → ['desensitising-fluoride', 'high-fluoride', 'standard-fluoride', 'tartar-control-fluoride']
  mouthwash_type           : 4 classes → ['antibacterial', 'antiseptic-chlorhexidine', 'fluoride-rinse', 'none']
  interdental_tool         : 4 classes → ['both', 'floss-only', 'interdental-brush-only', 'none']
  rinse_type               : 4 classes → ['chlorhexidine-rinse', 'hydrogen-peroxide-rinse', 'none', 'salt-water']
  pain_relief              : 5 classes → ['none', 'nsaid-plus-topical', 'nsaid-type', 'paracetamol-type', 'topical-anaesthetic']
  dietary_action           : 4 classes → ['no-change', 'reduce-acid', 'reduce-sugar', 'reduce-sugar-and-acid']
  dentist_urgency          : 5 classes → ['1-week', '2-weeks', '3-months', '4-weeks', '6-months']
  monitoring_focus         : 5 classes → ['bleeding', 'multiple-symptoms', 'pain-and-swelling', 'routine', 'sensitivity']
  lifestyle_action         : 4 classes → ['alcohol-reduction', 'both-smoking-and-a

In [14]:
# ── Fig 6: Correlation heatmap of numeric features ───────────────────────────
# Helps spot multicollinearity. Highly correlated pairs carry redundant info.
numeric_cols = FEATURE_COLS + ["risk_score"]
corr = df_enc[numeric_cols].corr()

fig, ax = plt.subplots(figsize=(14, 11))
mask = np.triu(np.ones_like(corr, dtype=bool))   # only lower triangle
sns.heatmap(corr, mask=mask, annot=True, fmt=".2f", cmap="RdBu_r",
            center=0, linewidths=0.4, ax=ax, annot_kws={"size": 7})
ax.set_title("Feature Correlation Matrix", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig("outputs/graphs/06_correlation_heatmap.png", bbox_inches="tight")
plt.show()
print("Saved → outputs/graphs/06_correlation_heatmap.png")

Saved → outputs/graphs/06_correlation_heatmap.png


## ✂️ Section 5 — Train / Test Split

**80/20 split** with `random_state=42` for reproducibility.
- `X` → input features (same for all 3 models)
- `y_score` → continuous risk score (Model 1 — regression)
- `y_level` → categorical Low/Med/High (Model 2 — classification)
- `y_care` → 9 care plan columns (Model 3 — multi-output classification)


In [15]:
# ── Prepare feature matrix and all targets ───────────────────────────────────
X       = df_enc[FEATURE_COLS]                            # shape (10000, 24)
y_score = df_enc["risk_score"]                            # continuous 0–100
y_level = df_enc["risk_level_enc"]                        # 0=low, 1=medium, 2=high
y_care  = df_enc[TARGET_ENC_COLS]                         # 9 integer columns

print(f"X shape      : {X.shape}")
print(f"y_score shape: {y_score.shape}  (regression target)")
print(f"y_level shape: {y_level.shape}  (classification target)")
print(f"y_care shape : {y_care.shape}   (multi-output target)")

# ── 80/20 split — same indices used for all three tasks ──────────────────────
# stratify on y_level ensures all 3 classes are proportionally represented
# in both train and test sets — avoids accidentally missing 'high' risk in test.
X_train, X_test, ys_train, ys_test, yl_train, yl_test, yc_train, yc_test = train_test_split(
    X, y_score, y_level, y_care,
    test_size=0.2, random_state=42, stratify=y_level
)

print(f"\nTrain size : {X_train.shape[0]} rows")
print(f"Test size  : {X_test.shape[0]} rows")
print(f"\nTrain risk_level distribution:")
print(pd.Series(yl_train).map({0:'low',1:'medium',2:'high'}).value_counts())

X shape      : (10000, 24)
y_score shape: (10000,)  (regression target)
y_level shape: (10000,)  (classification target)
y_care shape : (10000, 9)   (multi-output target)

Train size : 8000 rows
Test size  : 2000 rows

Train risk_level distribution:
risk_level_enc
medium    4525
low       1841
high      1634
Name: count, dtype: int64


In [16]:
# ── Apply SMOTE only to Model 2 training data ────────────────────────────────
# SMOTE (Synthetic Minority Over-sampling Technique) creates synthetic samples
# for the minority class so the classifier doesn't bias toward 'medium' (majority).
# We only apply it to Model 2 (classification) — NOT regression or care plan.

sm = SMOTE(random_state=42)
X_train_sm, yl_train_sm = sm.fit_resample(X_train, yl_train)

print("Before SMOTE — class distribution:")
for k, v in pd.Series(yl_train).value_counts().items():
    print(f"   {['low','medium','high'][k]:<8}: {v}")

print("\nAfter SMOTE — class distribution:")
for k, v in pd.Series(yl_train_sm).value_counts().items():
    print(f"   {['low','medium','high'][k]:<8}: {v}")
print("\nAll classes balanced. X_train_sm used only for Model 2.")

Before SMOTE — class distribution:
   medium  : 4525
   low     : 1841
   high    : 1634

After SMOTE — class distribution:
   low     : 4525
   medium  : 4525
   high    : 4525

All classes balanced. X_train_sm used only for Model 2.


## 📈 Section 6 — Model 1: Risk Score Regression

**Task:** Predict the continuous `risk_score` (0–100) from patient features.

**Why XGBoost?** It handles non-linear relationships, mixed feature types, and produces well-calibrated outputs. We compare it against Linear Regression and Random Forest.

**Metrics used:**
- **RMSE** — penalises large errors more (sensitive to outliers)
- **MAE** — average absolute error in score units
- **R²** — proportion of variance explained (1.0 = perfect)


In [17]:
# ── Define regression models to compare ──────────────────────────────────────
# We train three models and pick the best one by R² on the test set.

regression_models = {
    "Linear Regression" : LinearRegression(),
    "Random Forest"     : RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1),
    "XGBoost"           : XGBRegressor(
                              n_estimators=300,        # number of boosting rounds
                              learning_rate=0.05,      # smaller = more robust, needs more rounds
                              max_depth=6,             # tree depth — controls overfitting
                              subsample=0.8,           # row sampling per round
                              colsample_bytree=0.8,    # feature sampling per round
                              random_state=42,
                              verbosity=0
                          ),
}

reg_results = {}

for name, model in regression_models.items():
    model.fit(X_train, ys_train)
    preds = model.predict(X_test)

    mse = mean_squared_error(ys_test, preds)
    rmse = np.sqrt(mse)
    mae = mean_absolute_error(ys_test, preds)
    r2 = r2_score(ys_test, preds)

    reg_results[name] = {
        "RMSE": rmse,
        "MAE": mae,
        "R2": r2,
        "preds": preds,
        "model": model
    }

    print(f"{name:<25} | RMSE: {rmse:.3f}  MAE: {mae:.3f}  R²: {r2:.4f}")

# Pick best model by R²
best_reg_name  = max(reg_results, key=lambda k: reg_results[k]["R2"])
best_reg_model = reg_results[best_reg_name]["model"]
print(f"\n✅ Best regression model: {best_reg_name}  (R² = {reg_results[best_reg_name]['R2']:.4f})")

Linear Regression         | RMSE: 4.114  MAE: 3.308  R²: 0.9481
Random Forest             | RMSE: 4.148  MAE: 3.204  R²: 0.9473
XGBoost                   | RMSE: 2.376  MAE: 1.873  R²: 0.9827

✅ Best regression model: XGBoost  (R² = 0.9827)


In [18]:
# ── Fig 7: Regression model comparison ───────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

metrics = ["RMSE", "MAE", "R2"]
titles  = ["RMSE (lower is better)", "MAE (lower is better)", "R² (higher is better)"]

for ax, metric, title in zip(axes, metrics, titles):
    vals = [reg_results[n][metric] for n in regression_models]
    bars = ax.bar(regression_models.keys(), vals,
                  color=["#3498db","#2ecc71","#e74c3c"], edgecolor="white", alpha=0.85)
    ax.set_title(title, fontweight="bold")
    ax.set_ylabel(metric)
    for bar, v in zip(bars, vals):
        ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.002,
                f"{v:.3f}", ha="center", va="bottom", fontsize=9)
    ax.tick_params(axis="x", rotation=15)

plt.suptitle("Model 1: Risk Score Regression — Model Comparison", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig("outputs/graphs/07_regression_model_comparison.png", bbox_inches="tight")
plt.show()

In [19]:
# ── Fig 8: Actual vs Predicted scatter (best model) ──────────────────────────
# Dots should lie along the red dashed diagonal for a perfect model.
# Spread around the line = prediction error.

best_preds = reg_results[best_reg_name]["preds"]

fig, ax = plt.subplots(figsize=(7, 6))
ax.scatter(ys_test, best_preds, alpha=0.2, s=8, color="#2980b9")
lims = [0, 100]
ax.plot(lims, lims, "r--", linewidth=1.5, label="Perfect prediction")
ax.set_xlim(lims); ax.set_ylim(lims)
ax.set_xlabel("Actual Risk Score")
ax.set_ylabel("Predicted Risk Score")
ax.set_title(f"Model 1 — Actual vs Predicted ({best_reg_name})", fontweight="bold")
ax.legend()
plt.tight_layout()
plt.savefig("outputs/graphs/08_actual_vs_predicted.png", bbox_inches="tight")
plt.show()

In [20]:
# ── Fig 9: Feature importance for XGBoost regressor ──────────────────────────
# Shows which input features most strongly drive the risk score prediction.
if best_reg_name == "XGBoost":
    feat_imp = pd.Series(best_reg_model.feature_importances_, index=FEATURE_COLS)
else:
    feat_imp = pd.Series(best_reg_model.feature_importances_, index=FEATURE_COLS)

feat_imp = feat_imp.sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(9, 8))
feat_imp.plot(kind="barh", ax=ax, color="#e74c3c", alpha=0.85, edgecolor="white")
ax.set_title(f"Model 1 — Feature Importances ({best_reg_name})", fontweight="bold")
ax.set_xlabel("Importance Score")
plt.tight_layout()
plt.savefig("outputs/graphs/09_regression_feature_importance.png", bbox_inches="tight")
plt.show()

## 🌲 Section 7 — Model 2: Risk Level Classification

**Task:** Classify each patient as `Low`, `Medium`, or `High` risk.

**Input:** Same 24 features, but trained on SMOTE-balanced data to avoid bias toward the majority class (`medium`).

**Main model:** Random Forest — robust to noise, gives feature importances, low hyperparameter sensitivity.


In [22]:
# ── Define classification models to compare ──────────────────────────────────
clf_models = {
    "Logistic Regression": LogisticRegression(
        max_iter=1000,
        random_state=42
    ),

    "Decision Tree": DecisionTreeClassifier(
        max_depth=10,
        random_state=42
    ),

    "XGBoost": XGBClassifier(
        n_estimators=200,
        learning_rate=0.1,
        max_depth=5,
        use_label_encoder=False,
        eval_metric="mlogloss",
        random_state=42,
        verbosity=0
    ),

    "Random Forest": RandomForestClassifier(
        n_estimators=200,
        max_depth=12,
        random_state=42,
        n_jobs=-1
    ),
}

clf_results = {}

for name, model in clf_models.items():
    # Train on SMOTE-balanced data so all 3 classes are equally represented
    model.fit(X_train_sm, yl_train_sm)
    preds = model.predict(X_test)                   # evaluate on original test set

    acc = accuracy_score(yl_test, preds)
    f1  = f1_score(yl_test, preds, average="weighted")

    clf_results[name] = {"Accuracy": acc, "F1": f1, "preds": preds, "model": model}
    print(f"{name:<25} | Accuracy: {acc:.4f}  F1 (weighted): {f1:.4f}")

best_clf_name  = max(clf_results, key=lambda k: clf_results[k]["F1"])
best_clf_model = clf_results[best_clf_name]["model"]
print(f"\n✅ Best classification model: {best_clf_name}  (F1 = {clf_results[best_clf_name]['F1']:.4f})")

Logistic Regression       | Accuracy: 0.8665  F1 (weighted): 0.8674
Decision Tree             | Accuracy: 0.8230  F1 (weighted): 0.8251
XGBoost                   | Accuracy: 0.8880  F1 (weighted): 0.8887
Random Forest             | Accuracy: 0.8475  F1 (weighted): 0.8493

✅ Best classification model: XGBoost  (F1 = 0.8887)


In [23]:
# ── Fig 10: Classification model comparison ───────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for ax, metric in zip(axes, ["Accuracy", "F1"]):
    vals  = [clf_results[n][metric] for n in clf_models]
    bars  = ax.bar(clf_models.keys(), vals,
                   color=["#9b59b6","#3498db","#e67e22","#2ecc71"], edgecolor="white", alpha=0.85)
    ax.set_ylim(0, 1.1)
    ax.set_title(f"{metric} — Higher is Better", fontweight="bold")
    ax.set_ylabel(metric)
    for bar, v in zip(bars, vals):
        ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.01,
                f"{v:.3f}", ha="center", va="bottom", fontsize=9)
    ax.tick_params(axis="x", rotation=15)

plt.suptitle("Model 2: Risk Level Classification — Model Comparison", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig("outputs/graphs/10_classification_model_comparison.png", bbox_inches="tight")
plt.show()

In [24]:
# ── Fig 11: Confusion matrix for best classifier ─────────────────────────────
# Each cell shows how many test samples were predicted correctly / incorrectly.
# Diagonal = correct predictions. Off-diagonal = misclassifications.
best_clf_preds = clf_results[best_clf_name]["preds"]
labels         = ["Low", "Medium", "High"]
cm             = confusion_matrix(yl_test, best_clf_preds)

fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=labels,
            yticklabels=labels, linewidths=0.5, ax=ax)
ax.set_xlabel("Predicted Label")
ax.set_ylabel("True Label")
ax.set_title(f"Model 2 — Confusion Matrix ({best_clf_name})", fontweight="bold")
plt.tight_layout()
plt.savefig("outputs/graphs/11_confusion_matrix.png", bbox_inches="tight")
plt.show()

print("\nClassification Report:")
print(classification_report(yl_test, best_clf_preds, target_names=labels))


Classification Report:
              precision    recall  f1-score   support

         Low       0.91      0.89      0.90       460
      Medium       0.91      0.89      0.90      1132
        High       0.80      0.89      0.84       408

    accuracy                           0.89      2000
   macro avg       0.88      0.89      0.88      2000
weighted avg       0.89      0.89      0.89      2000



In [25]:
# ── Fig 12: Feature importance for best classifier ────────────────────────────
if hasattr(best_clf_model, "feature_importances_"):
    clf_imp = pd.Series(best_clf_model.feature_importances_, index=FEATURE_COLS).sort_values(ascending=True)

    fig, ax = plt.subplots(figsize=(9, 8))
    clf_imp.plot(kind="barh", ax=ax, color="#27ae60", alpha=0.85, edgecolor="white")
    ax.set_title(f"Model 2 — Feature Importances ({best_clf_name})", fontweight="bold")
    ax.set_xlabel("Importance Score")
    plt.tight_layout()
    plt.savefig("outputs/graphs/12_classification_feature_importance.png", bbox_inches="tight")
    plt.show()

## 🏥 Section 8 — Model 3: Care Plan Prediction (LightGBM MultiOutput)

**Task:** Simultaneously predict all **9 care plan decisions** from patient features.

**Why `MultiOutputClassifier`?** It wraps any single-output classifier and trains one model per target. Each of the 9 care plan columns is predicted independently.

**Why LightGBM?** Faster than XGBoost on multi-class problems, handles categorical-derived features well, and scales efficiently.

**Note:** `risk_score` is **not** used as an input here — the model learns care plans directly from clinical features. This makes it usable before Model 1 runs.


In [26]:
# ── Define care plan models to compare ───────────────────────────────────────
# MultiOutputClassifier wraps a base classifier to predict all 9 targets at once.
# Each target gets its own independent model internally.

care_models = {
    "Random Forest" : MultiOutputClassifier(
                          RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
                      ),
    "XGBoost"       : MultiOutputClassifier(
                          XGBClassifier(n_estimators=150, learning_rate=0.1, max_depth=5,
                                        use_label_encoder=False, eval_metric="mlogloss",
                                        random_state=42, verbosity=0)
                      ),
    "LightGBM"      : MultiOutputClassifier(
                          LGBMClassifier(n_estimators=200, learning_rate=0.05, max_depth=6,
                                         num_leaves=31, random_state=42, verbose=-1)
                      ),
}

care_results = {}

for name, model in care_models.items():
    model.fit(X_train, yc_train)            # train on original data (no SMOTE for multi-output)
    preds = model.predict(X_test)           # returns shape (n_test, 9)

    # Per-target accuracy: accuracy for each of the 9 care plan outputs separately
    per_target_acc = [accuracy_score(yc_test.iloc[:, i], preds[:, i])
                      for i in range(len(TARGET_COLS))]

    # Overall accuracy: all 9 predictions correct for a single patient
    exact_match = np.mean(np.all(preds == yc_test.values, axis=1))

    care_results[name] = {
        "mean_accuracy"  : np.mean(per_target_acc),
        "exact_match"    : exact_match,
        "per_target_acc" : per_target_acc,
        "preds"          : preds,
        "model"          : model,
    }

    print(f"{name:<15} | Mean per-target acc: {np.mean(per_target_acc):.4f} | "
          f"Exact match: {exact_match:.4f}")

best_care_name  = max(care_results, key=lambda k: care_results[k]["mean_accuracy"])
best_care_model = care_results[best_care_name]["model"]
print(f"\n✅ Best care plan model: {best_care_name}  "
      f"(mean accuracy = {care_results[best_care_name]['mean_accuracy']:.4f})")

Random Forest   | Mean per-target acc: 0.9839 | Exact match: 0.8755
XGBoost         | Mean per-target acc: 0.9886 | Exact match: 0.9075
LightGBM        | Mean per-target acc: 0.9878 | Exact match: 0.9035

✅ Best care plan model: XGBoost  (mean accuracy = 0.9886)


In [27]:
# ── Fig 13: Per-target accuracy across all models ─────────────────────────────
# Shows which care plan outputs are hardest to predict and which model wins.
fig, ax = plt.subplots(figsize=(14, 6))

x      = np.arange(len(TARGET_COLS))
width  = 0.25
colors = ["#3498db", "#e67e22", "#2ecc71"]

for i, (name, color) in enumerate(zip(care_models.keys(), colors)):
    offset = (i - 1) * width
    bars   = ax.bar(x + offset, care_results[name]["per_target_acc"],
                    width, label=name, color=color, edgecolor="white", alpha=0.85)

ax.set_xticks(x)
ax.set_xticklabels([t.replace("_", "\n") for t in TARGET_COLS], fontsize=9)
ax.set_ylim(0, 1.15)
ax.set_ylabel("Accuracy")
ax.set_title("Model 3 — Per-Target Accuracy for All Care Plan Outputs", fontweight="bold")
ax.legend()
ax.axhline(0.9, color="red", linestyle="--", linewidth=1, label="0.90 threshold")

plt.tight_layout()
plt.savefig("outputs/graphs/13_careplan_per_target_accuracy.png", bbox_inches="tight")
plt.show()

In [28]:
# ── Fig 14: Predicted label count distribution ────────────────────────────────
# Sanity check: the model should predict a realistic number of correct outputs
# per patient. Compare predicted vs actual number of exactly-matched outputs.

best_preds_care = care_results[best_care_name]["preds"]

# Count how many of the 9 targets were predicted correctly per patient
correct_per_patient = np.sum(best_preds_care == yc_test.values, axis=1)

fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(correct_per_patient, bins=range(0, 11), color="#8e44ad", edgecolor="white",
        alpha=0.85, align="left")
ax.set_xticks(range(0, 10))
ax.set_xlabel("Number of Care Plan Outputs Correctly Predicted (out of 9)")
ax.set_ylabel("Number of Patients")
ax.set_title(f"Model 3 — Correct Predictions Per Patient ({best_care_name})", fontweight="bold")
ax.axvline(correct_per_patient.mean(), color="red", linestyle="--",
           label=f"Mean: {correct_per_patient.mean():.2f}/9")
ax.legend()
plt.tight_layout()
plt.savefig("outputs/graphs/14_careplan_correct_per_patient.png", bbox_inches="tight")
plt.show()
print(f"Mean correct predictions per patient: {correct_per_patient.mean():.2f} / 9")

Mean correct predictions per patient: 8.90 / 9


## 💾 Section 9 — Save All Models & Encoders

Save every trained artefact so they can be loaded by a backend API without retraining.


In [29]:
# ── Save Model 1 — Risk Score Regressor ──────────────────────────────────────
joblib.dump(best_reg_model, "outputs/models/model1_risk_score_regressor.pkl")
print(f"✅ Saved: model1_risk_score_regressor.pkl  ({best_reg_name})")

# ── Save Model 2 — Risk Level Classifier ─────────────────────────────────────
joblib.dump(best_clf_model, "outputs/models/model2_risk_level_classifier.pkl")
print(f"✅ Saved: model2_risk_level_classifier.pkl  ({best_clf_name})")

# ── Save Model 3 — Care Plan MultiOutput Classifier ──────────────────────────
joblib.dump(best_care_model, "outputs/models/model3_careplan_recommender.pkl")
print(f"✅ Saved: model3_careplan_recommender.pkl  ({best_care_name})")

# ── Save LabelEncoders for decoding care plan predictions ────────────────────
joblib.dump(label_encoders, "outputs/models/careplan_label_encoders.pkl")
print(f"✅ Saved: careplan_label_encoders.pkl  (9 encoders)")

# ── Save feature column order — backend must use exact same order ─────────────
with open("outputs/models/feature_columns.json", "w") as f:
    json.dump(FEATURE_COLS, f, indent=2)
print(f"✅ Saved: feature_columns.json  ({len(FEATURE_COLS)} features)")

# ── Save ordinal encoding maps for preprocessing at inference time ─────────────
with open("outputs/models/ordinal_maps.json", "w") as f:
    json.dump(ordinal_maps, f, indent=2)
print(f"✅ Saved: ordinal_maps.json")

# ── Save risk_level encoding map ───────────────────────────────────────────────
with open("outputs/models/risk_level_map.json", "w") as f:
    json.dump({"low":0,"medium":1,"high":2}, f)
print(f"✅ Saved: risk_level_map.json")

print("\n📁 All artefacts saved to outputs/models/")

✅ Saved: model1_risk_score_regressor.pkl  (XGBoost)
✅ Saved: model2_risk_level_classifier.pkl  (XGBoost)
✅ Saved: model3_careplan_recommender.pkl  (XGBoost)
✅ Saved: careplan_label_encoders.pkl  (9 encoders)
✅ Saved: feature_columns.json  (24 features)
✅ Saved: ordinal_maps.json
✅ Saved: risk_level_map.json

📁 All artefacts saved to outputs/models/


## 🔬 Section 10 — End-to-End Prediction Pipeline

A single `predict_patient()` function that:
1. Takes a raw patient dictionary (same fields as the dataset)
2. Encodes and engineers features identically to training
3. Runs all 3 models
4. Returns risk score, risk level, and all 9 care plan decisions


In [30]:
# ── Feature engineering helper — must mirror Section 4 exactly ───────────────
def preprocess_patient(profile: dict) -> pd.DataFrame:
    """
    Convert a raw patient profile dict into a model-ready feature DataFrame.
    The output column order must exactly match FEATURE_COLS used in training.
    """
    p = profile.copy()

    # 1. Ordinal encoding
    for col, mapping in ordinal_maps.items():
        p[col] = mapping[p[col]]

    # 2. Binary encoding
    p["identified_disease"] = int(p["identified_disease"] == "periodontal_bone_loss")
    for col in ["is_primary_teeth","diabetes_status","pregnancy_status",
                "gum_bleeding","tooth_sensitivity","calcium_or_vitamin_deficiency"]:
        p[col] = int(p[col])

    # 3. Derived features
    total = p["number_of_teeth"] + p["number_of_missing_teeth"]
    p["missing_teeth_ratio"]     = p["number_of_missing_teeth"] / max(total, 1)
    p["filled_teeth_ratio"]      = p["number_of_filled_teeth"]  / max(p["number_of_teeth"], 1)
    p["affected_teeth_ratio"]    = p["affected_teeth_count"]    / max(p["number_of_teeth"], 1)
    p["lifestyle_risk_score"]    = p["smoking_status"] + p["alcohol_usage"] + p["sugar_usage"]
    p["oral_hygiene_risk_score"] = (p["overall_oral_hygiene_level"] +
                                    (2 - p["brushing_frequency"]) +
                                    p["gum_bleeding"] + p["tooth_sensitivity"])
    p["medical_risk_score"]      = (p["diabetes_status"] + p["pregnancy_status"] +
                                    p["calcium_or_vitamin_deficiency"] + p["is_primary_teeth"])

    return pd.DataFrame([{col: p[col] for col in FEATURE_COLS}])


def predict_patient(profile: dict) -> dict:
    """
    Run all 3 models on a single patient profile.

    Parameters
    ----------
    profile : dict  — raw patient data (string/bool values as in the CSV)

    Returns
    -------
    dict with keys:
        risk_score   : float (0–100)
        risk_level   : str   ('low' / 'medium' / 'high')
        care_plan    : dict  (9 care plan decisions as human-readable strings)
    """
    X_new = preprocess_patient(profile)

    # Model 1 — predict continuous risk score
    risk_score = float(best_reg_model.predict(X_new)[0])
    risk_score = round(np.clip(risk_score, 0, 100), 2)

    # Model 2 — predict risk level category
    level_enc  = int(best_clf_model.predict(X_new)[0])
    risk_level = ["low", "medium", "high"][level_enc]

    # Model 3 — predict all 9 care plan outputs
    care_enc   = best_care_model.predict(X_new)[0]   # integer array of length 9
    care_plan  = {
        col: label_encoders[col].inverse_transform([enc])[0]
        for col, enc in zip(TARGET_COLS, care_enc)
    }

    return {
        "risk_score" : risk_score,
        "risk_level" : risk_level,
        "care_plan"  : care_plan,
    }


print("✅ predict_patient() function defined and ready.")

✅ predict_patient() function defined and ready.


In [31]:
# ── Test Patient 1 — High-risk adult with severe disease ─────────────────────
patient_1 = {
    "age"                          : 45,
    "number_of_teeth"              : 24,
    "number_of_missing_teeth"      : 4,
    "is_primary_teeth"             : False,
    "smoking_status"               : "high",
    "alcohol_usage"                : "medium",
    "sugar_usage"                  : "high",
    "brushing_frequency"           : 0,           # never brushes
    "diabetes_status"              : True,
    "pregnancy_status"             : False,
    "gum_bleeding"                 : True,
    "tooth_sensitivity"            : True,
    "calcium_or_vitamin_deficiency": True,
    "number_of_filled_teeth"       : 6,
    "overall_oral_hygiene_level"   : "poor",
    "identified_disease"           : "periodontal_bone_loss",
    "disease_severity_from_xray"   : "severe",
    "affected_teeth_count"         : 12,
}

result_1 = predict_patient(patient_1)

print("=" * 55)
print("  PATIENT 1 — High-Risk Adult")
print("=" * 55)
print(f"  Risk Score  : {result_1['risk_score']:.1f} / 100")
print(f"  Risk Level  : {result_1['risk_level'].upper()}")
print()
print("  Care Plan:")
for k, v in result_1["care_plan"].items():
    print(f"    {k:<25}: {v}")
print("=" * 55)

  PATIENT 1 — High-Risk Adult
  Risk Score  : 100.0 / 100
  Risk Level  : HIGH

  Care Plan:
    toothpaste_type          : tartar-control-fluoride
    mouthwash_type           : antiseptic-chlorhexidine
    interdental_tool         : both
    rinse_type               : chlorhexidine-rinse
    pain_relief              : nsaid-plus-topical
    dietary_action           : reduce-sugar
    dentist_urgency          : 1-week
    monitoring_focus         : multiple-symptoms
    lifestyle_action         : both-smoking-and-alcohol


In [32]:
# ── Test Patient 2 — Low-risk young adult ────────────────────────────────────
patient_2 = {
    "age"                          : 22,
    "number_of_teeth"              : 28,
    "number_of_missing_teeth"      : 0,
    "is_primary_teeth"             : False,
    "smoking_status"               : "no",
    "alcohol_usage"                : "no",
    "sugar_usage"                  : "medium",
    "brushing_frequency"           : 2,
    "diabetes_status"              : False,
    "pregnancy_status"             : False,
    "gum_bleeding"                 : False,
    "tooth_sensitivity"            : False,
    "calcium_or_vitamin_deficiency": False,
    "number_of_filled_teeth"       : 1,
    "overall_oral_hygiene_level"   : "good",
    "identified_disease"           : "dental_cavity",
    "disease_severity_from_xray"   : "mild",
    "affected_teeth_count"         : 1,
}

result_2 = predict_patient(patient_2)

print("=" * 55)
print("  PATIENT 2 — Low-Risk Young Adult")
print("=" * 55)
print(f"  Risk Score  : {result_2['risk_score']:.1f} / 100")
print(f"  Risk Level  : {result_2['risk_level'].upper()}")
print()
print("  Care Plan:")
for k, v in result_2["care_plan"].items():
    print(f"    {k:<25}: {v}")
print("=" * 55)

  PATIENT 2 — Low-Risk Young Adult
  Risk Score  : 16.9 / 100
  Risk Level  : LOW

  Care Plan:
    toothpaste_type          : standard-fluoride
    mouthwash_type           : fluoride-rinse
    interdental_tool         : none
    rinse_type               : none
    pain_relief              : none
    dietary_action           : reduce-sugar
    dentist_urgency          : 6-months
    monitoring_focus         : routine
    lifestyle_action         : none


In [33]:
# ── Test Patient 3 — Pregnant patient with moderate risk ─────────────────────
patient_3 = {
    "age"                          : 29,
    "number_of_teeth"              : 27,
    "number_of_missing_teeth"      : 1,
    "is_primary_teeth"             : False,
    "smoking_status"               : "no",
    "alcohol_usage"                : "no",
    "sugar_usage"                  : "medium",
    "brushing_frequency"           : 1,
    "diabetes_status"              : False,
    "pregnancy_status"             : True,
    "gum_bleeding"                 : True,
    "tooth_sensitivity"            : True,
    "calcium_or_vitamin_deficiency": True,
    "number_of_filled_teeth"       : 3,
    "overall_oral_hygiene_level"   : "moderate",
    "identified_disease"           : "dental_cavity",
    "disease_severity_from_xray"   : "moderate",
    "affected_teeth_count"         : 3,
}

result_3 = predict_patient(patient_3)

print("=" * 55)
print("  PATIENT 3 — Pregnant Patient (Moderate Risk)")
print("=" * 55)
print(f"  Risk Score  : {result_3['risk_score']:.1f} / 100")
print(f"  Risk Level  : {result_3['risk_level'].upper()}")
print()
print("  Care Plan:")
for k, v in result_3["care_plan"].items():
    print(f"    {k:<25}: {v}")
print("=" * 55)
print()
print("⚠️  DISCLAIMER: Synthetic research data only. Not for clinical use.")

  PATIENT 3 — Pregnant Patient (Moderate Risk)
  Risk Score  : 58.3 / 100
  Risk Level  : MEDIUM

  Care Plan:
    toothpaste_type          : desensitising-fluoride
    mouthwash_type           : fluoride-rinse
    interdental_tool         : floss-only
    rinse_type               : salt-water
    pain_relief              : paracetamol-type
    dietary_action           : reduce-sugar
    dentist_urgency          : 2-weeks
    monitoring_focus         : multiple-symptoms
    lifestyle_action         : none

⚠️  DISCLAIMER: Synthetic research data only. Not for clinical use.
